## EIRP Grid Search

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pylupnt as pnt

In [ ]:
## DO NOT EDIT BELOW THIS LINE ##
n_sma = 500
n_incs = 500
distance_threshold_km = 100

smas = (
    np.linspace(4000, 16000, n_sma) * 1000
)  # Semi-major axis range from 4,000 km to 16,000 km
critical_inc = np.arccos(np.sqrt(3 / 5))  # Critical inclination in radians
incs_op = np.linspace(
    critical_inc, np.deg2rad(70), n_incs
)  # Inclination range from critical inclination to 70 degrees
ecc_op = np.sqrt(1 - 5 / 3 * np.cos(incs_op) ** 2)  # Eccentricity for frozen orbits
et0 = pnt.convert_time(pnt.gregorian_to_time(2030, 1, 1, 12, 0, 0), pnt.UTC, pnt.TAI)
Omega = 0
w = np.deg2rad(90)  # Argument of perigee fixed at 90 degrees
M = np.deg2rad(180)  # Mean anomaly fixed at 180 degrees
n_sim = n_sma * n_incs

sma_grid, inc_grid = np.meshgrid(smas, incs_op)
ecc_grid = np.sqrt(1 - 5 / 3 * np.cos(inc_grid) ** 2)  # Eccentricity for frozen orbits
r_peri = sma_grid * (1 - ecc_grid)  # Periapsis distance
print("Size sma_grid:", sma_grid.shape)

# convert to Cartesian coordinates for distance calculation
coe_op = np.zeros((n_sim, 6))
coe_op[:, 0] = sma_grid.flatten()  # Semi-major axis
coe_op[:, 1] = ecc_grid.flatten()  # Eccentricity
coe_op[:, 2] = inc_grid.flatten()  # Inclination
coe_op[:, 3] = Omega  # RAAN
coe_op[:, 4] = w  # Argument of perigee
coe_op[:, 5] = M  # Mean anomaly

cart_op = pnt.classical_to_cart(coe_op, pnt.GM_MOON)
cart_pa = pnt.convert_frame(et0, cart_op, pnt.MOON_OP, pnt.MOON_PA)
r_dist = np.linalg.norm(
    cart_pa[:, 0:3] - np.array([0, 0, -pnt.R_MOON]), axis=1
)  # Distance from the surface of the Moon
r_dist_grid = r_dist.reshape(n_incs, n_sma)
r_dist_grid[r_peri < (pnt.R_MOON + distance_threshold_km)] = (
    np.nan
)  # Set distance to 0 for orbits that intersect the surface
r_peri[r_peri < (pnt.R_MOON + distance_threshold_km)] = (
    np.nan
)  # Set periapsis distance to NaN for orbits that intersect the surface

r_dist_max_km = np.max(r_dist_grid) / 1000  # Convert to km
print(f"Maximum distance from the surface of the Moon: {r_dist_max_km:.2f} km")

# plot the disance map
plt.figure(figsize=(6, 4))
plt.pcolormesh(
    np.rad2deg(inc_grid),
    sma_grid / 1000,
    r_dist_grid / 1000,
    shading="auto",
    cmap="viridis",
)
plt.colorbar(label="Apoapsis Distance (km)")
plt.xlabel("Inclination (degrees)")
plt.ylabel("Semi-major Axis (km)")
plt.title("Apoapsis Distance Map (to South-Pole) for Frozen Orbits around the Moon")
plt.grid()

# For each inclination, find the minimum sma to achieve a certain distance threshold (e.g., 100 km above the surface)
# sma_thresholds_km =  (1737.4 + distance_threshold_km) * np.ones(n_incs) / (1 - ecc_op)  # Calculate the required sma for the given eccentricity
# plt.plot(np.rad2deg(incs_op), sma_thresholds_km, 'r--', label='Distance Threshold (100 km above surface)')
plt.ylim(4000, 16000)

In [ ]:
C = 299792.458  # Speed of light in km/s

def get_eirp(r_dist_km, freq_Hz, P_rx_dB=-160.0):
    """
    Get the required transmit EIRP for a given distance and frequency
    r_dist in km, freq in Hz
    Returns EIRP in dBW
    """
    # Calculate wavelength in meters
    wavelength = C / freq_hz  # C is in km/s, so wavelength will be in km
    
    # Convert wavelength to meters for the FSPL formula
    L_fs = 20 * np.log10((4 * np.pi * r_dist_km) / wavelength)  # FSPL in dB

    G_rx_dB = 3.0. # Receiver antenna gain in dBi (e.g., for a small patch antenna)
    L_tx_dB = 1.0  # Transmitter losses (e.g., due to inefficiencies in the power amplifier, feedline losses, etc.)
    L_rx_dB = 1.0. # Receiver losses (e.g., due to inefficiencies in the antenna, feedline losses, etc.)
    L_m_dB = 0.0.  # Additional miscellaneous losses (e.g., atmospheric losses, polarization mismatch, etc.)

    P_tx_dB = P_rx_dB + L_tx_dB - G_rx_dB + L_rx_dB + L_m_dB + L_fs
    
    return P_tx_dB

In [ ]:
import matplotlib.colors as colors

fig_dir = "figs/eirp"
freq_hz = 2492.028e6  # S-band frequency in Hz
required_powers_160 = get_eirp(
    r_dist_grid / 1000, freq_hz, P_rx_dB=-160.0
)  # Required transmit power for -160 dBW at apogee

norm = colors.Normalize(vmin=5, vmax=45)
levels = np.arange(5, 46, 2.5)

# Figure 1: Required transmit power to achive -160 dBW at apoapsis
fig, ax = plt.subplots(figsize=(6, 4))
cf = ax.contourf(
    np.rad2deg(inc_grid),
    sma_grid / 1000,
    required_powers_160,
    levels=np.arange(5, 46, 2.5),
    cmap="viridis",
    norm=norm,
)
# add colorbar
cbar = plt.colorbar(cf, ax=ax, label="Required EIRP (dBW)", ticks=np.arange(5, 50, 5))
ax.set_ylabel("Semi-Major Axis (km)")
ax.set_xlabel("Inclination (degrees)")
ax.set_title("Required EIRP (-160 dBW at apoapsis)")
ax.grid()
plt.tight_layout()
plt.savefig(f"{fig_dir}/required_eirp_160.pdf", dpi=300)
plt.show()

# Figure 2: Required transmit EIRP to achive -147 dBW at apoapsis
fig, ax = plt.subplots(figsize=(6, 4))
required_powers_147 = get_eirp(
    r_dist_grid / 1000, freq_hz, P_rx_dB=-147.0
)  # Required transmit EIRP for -147 dBW at apogee
cf = ax.contourf(
    np.rad2deg(inc_grid),
    sma_grid / 1000,
    required_powers_147,
    levels=np.arange(5, 46, 2.5),
    cmap="viridis",
    norm=norm,
)
# add colorbar
cbar = plt.colorbar(cf, ax=ax, label="Required EIRP (dBW)", ticks=np.arange(5, 50, 5))
ax.set_ylabel("Semi-Major Axis (km)")
ax.set_xlabel("Inclination (degrees)")
ax.set_title("Required EIRP (-147 dBW at apoapsis)")
ax.grid()
plt.tight_layout()
plt.savefig(f"{fig_dir}/required_eirp_147.pdf", dpi=300)
plt.show()

# Figure 3: Apolune - Perilune distance map
fig, ax = plt.subplots(figsize=(6, 4))
apolune_perilune_diff = (
    r_dist_grid - (r_peri - 1737.4e3)
) / 1000  # Difference between apolune and perilune distances
print("apolune perilune diff (max, km):", np.nanmax(apolune_perilune_diff))
print("apolune perilune diff (min, km):", np.nanmin(apolune_perilune_diff))
C = 299792.458  # Speed of light in km/s
wavelength = C / freq_hz  # C is in km/s, so wavelength will be in km
L_fs_diff = 20 * np.log10((4 * np.pi * r_dist_grid) / wavelength) - 20 * np.log10(
    (4 * np.pi * (r_peri - 1737.4e3)) / wavelength
)  # FSPL difference in dB

cf = ax.contourf(
    np.rad2deg(inc_grid),
    sma_grid / 1000,
    L_fs_diff,
    cmap="viridis",
    levels=np.arange(0, 100, 1),
    vmin=0,
    vmax=30,
)
cbar = plt.colorbar(cf, ax=ax, label="Free Space Path Loss Difference (dB)")

# plot contour for 13 dB difference
contour = ax.contour(
    np.rad2deg(inc_grid),
    sma_grid / 1000,
    L_fs_diff,
    levels=[13],
    colors="red",
    linewidths=2,
    linestyles="--",
)
cbar.add_lines(contour)

ax.set_ylabel("Semi-Major Axis (km)")
ax.set_xlabel("Inclination (degrees)")
ax.set_xticks(np.arange(40, 70, 2))
ax.set_yticks(np.arange(4000, 16000, 1000))
ax.set_title("Free Space Path Loss Difference (Apolune - Perilune)")
ax.grid()
plt.tight_layout()
plt.savefig(f"{fig_dir}/free_space_path_loss_difference.pdf", dpi=300)
plt.show()

print(
    "Required EIRP for -160 dBW at apoapsis (min, dBW):", np.nanmin(required_powers_160)
)
print(
    "Required EIRP for -160 dBW at apoapsis (max, dBW):", np.nanmax(required_powers_160)
)
print(
    "Required EIRP for -147 dBW at apoapsis (min, dBW):", np.nanmin(required_powers_147)
)
print(
    "Required EIRP for -147 dBW at apoapsis (max, dBW):", np.nanmax(required_powers_147)
)